In [ ]:
# ==============================================================
# Coral Health Classification (Bleached vs Healthy)
# Using Transfer Learning: ResNet50 & EfficientNetB0
#
# Dataset structure:
# ├── bleached_corals/   (485 images)
# ├── healthy_corals/    (483 images)
# ==============================================================

import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# ==============================================================
# 1. SETUP & PARAMETERS
# ==============================================================

DATA_DIR = "./"   # Change this to your dataset path if needed
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

bleached_dir = os.path.join(DATA_DIR, "bleached_corals")
healthy_dir = os.path.join(DATA_DIR, "healthy_corals")

# ==============================================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================

bleached_images = os.listdir(bleached_dir)
healthy_images = os.listdir(healthy_dir)
print(f"Bleached coral images: {len(bleached_images)}")
print(f"Healthy coral images: {len(healthy_images)}")

# Plot few random samples
def show_samples(class_name, path, n=5):
    class_path = os.path.join(path, class_name)
    sample_files = random.sample(os.listdir(class_path), n)
    plt.figure(figsize=(15, 5))
    for i, file in enumerate(sample_files):
        img = Image.open(os.path.join(class_path, file))
        plt.subplot(1, n, i+1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis("off")
    plt.show()

show_samples("bleached_corals", DATA_DIR)
show_samples("healthy_corals", DATA_DIR)

# Class distribution
labels = ["bleached_corals"] * len(bleached_images) + ["healthy_corals"] * len(healthy_images)
sns.countplot(x=labels, palette="coolwarm")
plt.title("Class Distribution")
plt.show()

# ==============================================================
# 3. TRAIN-VALIDATION-TEST SPLIT
# ==============================================================

# Create dataframe for generator
data = []
for label, folder in enumerate(["bleached_corals", "healthy_corals"]):
    folder_path = os.path.join(DATA_DIR, folder)
    for img_name in os.listdir(folder_path):
        data.append((os.path.join(folder_path, img_name), folder))

df = pd.DataFrame(data, columns=["filepath", "label"])
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["label"], random_state=SEED)

print("Train:", train_df.shape, "Validation:", val_df.shape, "Test:", test_df.shape)

# ==============================================================
# 4. DATA AUGMENTATION & DATA GENERATORS
# ==============================================================

train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)
test_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_gen_resnet = train_datagen_resnet.flow_from_dataframe(
    train_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", seed=SEED)

val_gen_resnet = val_datagen_resnet.flow_from_dataframe(
    val_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", seed=SEED)

test_gen_resnet = test_datagen_resnet.flow_from_dataframe(
    test_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)

# ==============================================================
# 5. TRAIN RESNET50 MODEL
# ==============================================================

base_resnet = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
for layer in base_resnet.layers[:-20]:
    layer.trainable = False

model_resnet = Sequential([
    base_resnet,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])

model_resnet.compile(optimizer=Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])
history_resnet = model_resnet.fit(
    train_gen_resnet, validation_data=val_gen_resnet, epochs=10
)

# ==============================================================
# 6. TRAIN EFFICIENTNETB0 MODEL
# ==============================================================

train_datagen_effnet = ImageDataGenerator(
    preprocessing_function=effnet_preprocess,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen_effnet = ImageDataGenerator(preprocessing_function=effnet_preprocess)
test_datagen_effnet = ImageDataGenerator(preprocessing_function=effnet_preprocess)

train_gen_effnet = train_datagen_effnet.flow_from_dataframe(
    train_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", seed=SEED)

val_gen_effnet = val_datagen_effnet.flow_from_dataframe(
    val_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", seed=SEED)

test_gen_effnet = test_datagen_effnet.flow_from_dataframe(
    test_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)

base_effnet = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
for layer in base_effnet.layers[:-20]:
    layer.trainable = False

model_effnet = Sequential([
    base_effnet,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])

model_effnet.compile(optimizer=Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])
history_effnet = model_effnet.fit(
    train_gen_effnet, validation_data=val_gen_effnet, epochs=10
)

# ==============================================================
# 7. EVALUATION ON TEST SET
# ==============================================================

resnet_eval = model_resnet.evaluate(test_gen_resnet)
effnet_eval = model_effnet.evaluate(test_gen_effnet)

print(f"\nResNet50 Test Accuracy: {resnet_eval[1]:.4f}")
print(f"EfficientNetB0 Test Accuracy: {effnet_eval[1]:.4f}")

# ==============================================================
# 8. PLOT TRAINING HISTORY
# ==============================================================

def plot_history(history, title):
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='val')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.title(f'{title} Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

plot_history(history_resnet, "ResNet50")
plot_history(history_effnet, "EfficientNetB0")

# ==============================================================
# 9. COMPARISON SUMMARY
# ==============================================================

results = pd.DataFrame({
    "Model": ["ResNet50", "EfficientNetB0"],
    "Test Accuracy": [resnet_eval[1], effnet_eval[1]],
    "Test Loss": [resnet_eval[0], effnet_eval[0]]
})
print(results)
sns.barplot(x="Model", y="Test Accuracy", data=results, palette="viridis")
plt.title("Model Comparison - Test Accuracy")